# Causal Machine Learning — Iron Supplementation Benefit/Harm Prediction

**Objetivo:** Identificar QUAIS crianças se beneficiam e quais são prejudicadas pela suplementação de ferro, usando Causal Forest + SHAP + validação rigorosa.

**Metodologia baseada em:** Carvalho e Silva et al. (IJMI 2026) — mesma pipeline de validação.

**Produto final:** Modelo deployável (GitHub Pages) que orienta se uma criança deve ou não receber ferro.

---

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

from econml.dml import CausalForestDML
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.model_selection import train_test_split, cross_val_score
from scipy.stats import norm, pearsonr, spearmanr
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

print('Libs carregadas')

## 1. Carregar Dados Preparados do data1

In [ ]:
# Carregar e preparar (replica data1 cells 0-13)
csv_path = '/Users/marcelocarvalhoesilva/project/iron/data_crianca_calib_anon.csv'
DF = pd.read_csv(csv_path, low_memory=False)
DF['age_months'] = DF['b05a_idade_em_meses'].str.extract(r'(\d+)').astype(float)
DF = DF[(DF['age_months'] >= 6) & (DF['age_months'] <= 18)].copy()
DF['age_group'] = pd.cut(DF['age_months'], bins=[5, 11, 18], labels=['6-11m', '12-18m'])

# Exposição
DF['iron_any'] = (DF['vd_supl1_com_ferro'] == 'Sim').astype(int)

# Confounders
DF['male'] = (DF['b02_sexo'] == 'Masculino').astype(int)
DF['race_preta'] = (DF['d01_cor'] == 'Preta').astype(int)
DF['race_parda'] = DF['d01_cor'].isin(['Parda (mulata, cabocla, cafuza, mameluca)', 'Parda']).astype(int)
for r, code in [('Norte', 1), ('Nordeste', 2), ('Sul', 4)]:
    DF[f'reg_{r.lower()}'] = (DF['a00_regiao'] == code).astype(int)
DF['reg_co'] = (DF['a00_regiao'] == 5).astype(int)
DF['urban'] = (DF['a11_situacao'] == 'Urbano').astype(int) if DF['a11_situacao'].dtype == 'object' else (DF['a11_situacao'] == 1).astype(int)
DF['educ_raw'] = pd.to_numeric(DF['j10_serie'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
DF['educ_fund_comp'] = ((DF['educ_raw'] >= 9) & (DF['educ_raw'] < 12)).astype(int)
DF['educ_medio'] = (DF['educ_raw'] == 12).astype(int)
DF['educ_superior'] = (DF['educ_raw'] > 12).astype(int)
DF['ien'] = pd.to_numeric(DF['vd_ien_quintos'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
DF['ebia_leve'] = (DF['vd_ebia_categ'] == 'Insegurança leve').astype(int) if DF['vd_ebia_categ'].dtype == 'object' else 0
DF['ebia_mod'] = (DF['vd_ebia_categ'] == 'Insegurança moderada').astype(int) if DF['vd_ebia_categ'].dtype == 'object' else 0
DF['ebia_grave'] = (DF['vd_ebia_categ'] == 'Insegurança grave').astype(int) if DF['vd_ebia_categ'].dtype == 'object' else 0
DF['esgoto_adeq'] = DF['p10_esgoto'].isin(['Rede geral de esgoto ou pluvial', 'Fossa séptica ligada a rede', 'Fossa séptica não ligada a rede', 1, 2, 3]).astype(int)
DF['agua_rede'] = DF['p11_agua'].isin(['Rede geral de distribuição', 1]).astype(int)
DF['cesarean'] = DF['h04_parto'].isin(['Cesariana agendada (eletiva)', 'Cesariana de urgência (não agendada)', 2, 3]).astype(int)
DF['gest_weeks'] = pd.to_numeric(DF['h01_semanas_gravidez'], errors='coerce')
DF['birth_weight'] = pd.to_numeric(DF['h02_peso'], errors='coerce')
DF['prenatal_visits'] = pd.to_numeric(DF['k05_prenatal_consultas'], errors='coerce')
DF['prenatal_visits'] = DF['prenatal_visits'].fillna(DF['prenatal_visits'].median())
DF['maternal_age'] = pd.to_numeric(DF['bb04_idade_da_mae'], errors='coerce')
DF['breastfed'] = (DF['e01_leite_peito'] == 'Sim').astype(int) if DF['e01_leite_peito'].dtype == 'object' else (DF['e01_leite_peito'] == 1).astype(int)
DF['formula'] = (DF['e10_formula_infantil'] == 'Sim').astype(int) if DF['e10_formula_infantil'].dtype == 'object' else (DF['e10_formula_infantil'] == 1).astype(int)
DF['cow_milk'] = ((DF['e06_leite_vaca_po'].isin(['Sim', 1])) | (DF['e07_leite_vaca_liquido'].isin(['Sim', 1]))).astype(int)
DF['feed_exclusive'] = ((DF['breastfed'] == 1) & (DF['formula'] == 0) & (DF['cow_milk'] == 0)).astype(int)
DF['feed_mixed'] = ((DF['breastfed'] == 1) & ((DF['formula'] == 1) | (DF['cow_milk'] == 1))).astype(int)

print(f'DF: {DF.shape[0]} crianças, {DF.shape[1]} colunas')
print(f'Ferro: {DF["iron_any"].sum()} ({100*DF["iron_any"].mean():.1f}%)')

## 2. Definição de Features e Outcomes

In [ ]:
# Features para o modelo causal (características da criança)
FEATURES = [
    'age_months', 'male', 'race_preta', 'race_parda',
    'reg_norte', 'reg_nordeste', 'reg_sul', 'reg_co', 'urban',
    'educ_fund_comp', 'educ_medio', 'educ_superior', 'ien',
    'ebia_leve', 'ebia_mod', 'ebia_grave',
    'esgoto_adeq', 'agua_rede', 'cesarean',
    'gest_weeks', 'birth_weight', 'prenatal_visits', 'maternal_age',
    'breastfed', 'formula', 'feed_exclusive', 'feed_mixed',
]

FEATURE_NAMES = [
    'Idade (meses)', 'Sexo masculino', 'Cor preta', 'Cor parda',
    'Norte', 'Nordeste', 'Sul', 'Centro-Oeste', 'Urbano',
    'Educ: Fund.completo', 'Educ: Médio', 'Educ: Superior', 'IEN (quintil)',
    'EBIA: leve', 'EBIA: moderada', 'EBIA: grave',
    'Saneamento adequado', 'Água rede', 'Cesariana',
    'Idade gestacional', 'Peso nascer', 'Consultas PN', 'Idade materna',
    'Aleitamento materno', 'Fórmula infantil', 'Aleit. exclusivo', 'Aleit. misto',
]

OUTCOMES = {
    'vd_anthro_zwfl': 'Z peso/comprimento (WFL)',
    'vd_hb_final': 'Hemoglobina (g/dL)',
}

print(f'Features: {len(FEATURES)}')
print(f'Outcomes: {list(OUTCOMES.values())}')

## 3. Train-Test Split (80/20) com Verificação de Integridade

Seguindo Carvalho e Silva (IJMI 2026):
- Split estratificado por tratamento
- Verificação de zero overlap de IDs
- Distribuição idêntica de target nos dois sets
- Hold-out isolado de todo preprocessing

In [ ]:
results_all = {}

for outcome_col, outcome_name in OUTCOMES.items():
    print(f"\n{'#'*80}")
    print(f'# OUTCOME: {outcome_name}')
    print(f"{'#'*80}")
    
    # Preparar dados completos
    cols = ['iron_any', outcome_col] + FEATURES
    data = DF[cols].dropna().copy()
    
    # Split estratificado
    train_idx, test_idx = train_test_split(
        data.index, test_size=0.2, random_state=42, stratify=data['iron_any']
    )
    train = data.loc[train_idx]
    test = data.loc[test_idx]
    
    # Verificações de integridade (IJMI padrão)
    id_overlap = set(train_idx).intersection(set(test_idx))
    train_iron_pct = train['iron_any'].mean() * 100
    test_iron_pct = test['iron_any'].mean() * 100
    
    print(f'\n--- Verificação de Integridade ---')
    print(f'  Total: {len(data)} | Train: {len(train)} ({100*len(train)/len(data):.1f}%) | Test: {len(test)} ({100*len(test)/len(data):.1f}%)')
    print(f'  ID overlap: {len(id_overlap)} (deve ser 0) {"✓" if len(id_overlap)==0 else "✗ ERRO"}')
    print(f'  Iron train: {train_iron_pct:.1f}% | Iron test: {test_iron_pct:.1f}% | Diff: {abs(train_iron_pct-test_iron_pct):.2f}% {"✓" if abs(train_iron_pct-test_iron_pct)<1 else "✗"}')
    print(f'  Outcome train: {train[outcome_col].mean():.3f}±{train[outcome_col].std():.3f}')
    print(f'  Outcome test:  {test[outcome_col].mean():.3f}±{test[outcome_col].std():.3f}')
    
    results_all[outcome_col] = {
        'data': data, 'train': train, 'test': test,
        'train_idx': train_idx, 'test_idx': test_idx,
    }

## 4. Comparação de Múltiplos Modelos Causais

Análogo à comparação de 9 algoritmos do IJMI paper, comparamos 3 meta-learners causais:
1. **CausalForestDML** (Athey & Imbens) — principal
2. **T-Learner** (dois modelos separados: E[Y|X,T=1] e E[Y|X,T=0])
3. **S-Learner** (um modelo com T como feature)

In [ ]:
from econml.metalearners import TLearner, SLearner

for outcome_col, outcome_name in OUTCOMES.items():
    print(f"\n{'='*80}")
    print(f'COMPARAÇÃO DE MODELOS CAUSAIS — {outcome_name}')
    print(f"{'='*80}")
    
    r = results_all[outcome_col]
    train, test = r['train'], r['test']
    
    Y_train = train[outcome_col].values
    T_train = train['iron_any'].values
    X_train = train[FEATURES].values
    Y_test = test[outcome_col].values
    T_test = test['iron_any'].values
    X_test = test[FEATURES].values
    
    models = {}
    
    # 1. CausalForestDML
    print('\n  Treinando CausalForestDML...')
    cf = CausalForestDML(
        model_y=GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42),
        model_t=GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42),
        n_estimators=1000, min_samples_leaf=30, random_state=42, verbose=0
    )
    cf.fit(Y_train, T_train, X=X_train)
    cate_cf_train = cf.effect(X_train)
    cate_cf_test = cf.effect(X_test)
    models['CausalForestDML'] = {'model': cf, 'cate_train': cate_cf_train, 'cate_test': cate_cf_test}
    
    # 2. T-Learner
    print('  Treinando T-Learner...')
    tl = TLearner(models=GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42))
    tl.fit(Y_train, T_train, X=X_train)
    cate_tl_train = tl.effect(X_train)
    cate_tl_test = tl.effect(X_test)
    models['T-Learner'] = {'model': tl, 'cate_train': cate_tl_train, 'cate_test': cate_tl_test}
    
    # 3. S-Learner
    print('  Treinando S-Learner...')
    sl = SLearner(overall_model=GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42))
    sl.fit(Y_train, T_train, X=X_train)
    cate_sl_train = sl.effect(X_train)
    cate_sl_test = sl.effect(X_test)
    models['S-Learner'] = {'model': sl, 'cate_train': cate_sl_train, 'cate_test': cate_sl_test}
    
    # Comparar ATEs
    print(f'\n  --- ATE Comparação (holdout) ---')
    print(f'  {"Modelo":<20} {"ATE train":>12} {"ATE test":>12} {"Diff":>8} {"Corr train-test":>15}')
    print(f'  {"-"*67}')
    for name, m in models.items():
        ate_tr = m['cate_train'].mean()
        ate_te = m['cate_test'].mean()
        corr, _ = pearsonr(m['cate_train'][:len(m['cate_test'])], m['cate_test'][:len(m['cate_train'])]) if len(m['cate_train']) == len(m['cate_test']) else (np.nan, np.nan)
        print(f'  {name:<20} {ate_tr:>12.4f} {ate_te:>12.4f} {abs(ate_tr-ate_te):>8.4f}')
    
    # Concordância entre modelos (correlação dos CATEs no test)
    print(f'\n  --- Concordância entre modelos (Pearson r no test set) ---')
    model_names = list(models.keys())
    for i in range(len(model_names)):
        for j in range(i+1, len(model_names)):
            c1 = models[model_names[i]]['cate_test'].flatten()
            c2 = models[model_names[j]]['cate_test'].flatten()
            r_val, p_val = pearsonr(c1, c2)
            print(f'  {model_names[i]} vs {model_names[j]}: r={r_val:.3f} (p={p_val:.4f})')
    
    r['models'] = models
    print('\n  ✓ Modelos treinados e comparados')

## 5. Bootstrap CI (n=1000) para CATE por Subgrupo

Seguindo IJMI paper: 1000 iterações bootstrap para IC 95% de cada estimativa.

In [ ]:
N_BOOT = 1000

def bootstrap_cate(cate_values, n_boot=N_BOOT):
    """Bootstrap 95% CI for mean CATE."""
    boot_means = []
    for _ in range(n_boot):
        idx = np.random.choice(len(cate_values), len(cate_values), replace=True)
        boot_means.append(cate_values[idx].mean())
    return np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)

for outcome_col, outcome_name in OUTCOMES.items():
    print(f"\n{'='*80}")
    print(f'BOOTSTRAP CI (n={N_BOOT}) — {outcome_name}')
    print(f"{'='*80}")
    
    r = results_all[outcome_col]
    cf_model = r['models']['CausalForestDML']
    test = r['test'].copy()
    test['cate'] = cf_model['cate_test']
    
    # ATE global
    cate_test = cf_model['cate_test'].flatten()
    ci_lo, ci_hi = bootstrap_cate(cate_test)
    print(f'\n  ATE global: {cate_test.mean():.4f} (95% CI: {ci_lo:.4f} to {ci_hi:.4f})')
    
    # Subgrupos
    subgroups = {
        'Não amamentado': test['breastfed'] == 0,
        'Amamentado': test['breastfed'] == 1,
        'Aleit. exclusivo': test['feed_exclusive'] == 1,
        'Aleit. misto': test['feed_mixed'] == 1,
        'Idade 6-11m': test['age_months'] <= 11,
        'Idade 12-18m': test['age_months'] > 11,
        'Não amam 6-11m': (test['breastfed'] == 0) & (test['age_months'] <= 11),
        'Não amam 12-18m': (test['breastfed'] == 0) & (test['age_months'] > 11),
        'Amam 6-11m': (test['breastfed'] == 1) & (test['age_months'] <= 11),
        'Amam 12-18m': (test['breastfed'] == 1) & (test['age_months'] > 11),
        'IEN Q1 (mais pobre)': test['ien'] == 1,
        'IEN Q5 (mais rico)': test['ien'] == 5,
    }
    
    print(f'\n  {"Subgrupo":<25} {"N":>5} {"CATE":>8} {"95% CI":>20} {"Sig":>5}')
    print(f'  {"-"*63}')
    for name, mask in subgroups.items():
        if mask.sum() < 10:
            continue
        cate_sub = test.loc[mask, 'cate'].values.flatten()
        ci_lo, ci_hi = bootstrap_cate(cate_sub)
        sig = '*' if (ci_lo > 0 or ci_hi < 0) else ''
        print(f'  {name:<25} {mask.sum():>5} {cate_sub.mean():>8.4f} ({ci_lo:>8.4f} to {ci_hi:>8.4f}) {sig:>5}')

## 6. GATES — Group Average Treatment Effects

Divide crianças em quintis de CATE predito e verifica se o efeito real difere entre quintis.
Se Q1 (mais prejudicadas) tem efeito significativamente diferente de Q5 (mais beneficiadas), a heterogeneidade é real.

In [ ]:
for outcome_col, outcome_name in OUTCOMES.items():
    print(f"\n{'='*80}")
    print(f'GATES — {outcome_name}')
    print(f"{'='*80}")
    
    r = results_all[outcome_col]
    test = r['test'].copy()
    cate_test = r['models']['CausalForestDML']['cate_test'].flatten()
    test['cate'] = cate_test
    
    # Quintis de CATE predito
    test['cate_quintile'] = pd.qcut(test['cate'], q=5, labels=['Q1 (menor)', 'Q2', 'Q3', 'Q4', 'Q5 (maior)'])
    
    print(f'\n  {"Quintil CATE":<15} {"N":>5} {"CATE médio":>12} {"Y ferro":>10} {"Y sem ferro":>12} {"Diff real":>10}')
    print(f'  {"-"*64}')
    
    for q in ['Q1 (menor)', 'Q2', 'Q3', 'Q4', 'Q5 (maior)']:
        mask = test['cate_quintile'] == q
        sub = test[mask]
        y_iron = sub.loc[sub['iron_any'] == 1, outcome_col].mean()
        y_no_iron = sub.loc[sub['iron_any'] == 0, outcome_col].mean()
        diff_real = y_iron - y_no_iron
        print(f'  {q:<15} {mask.sum():>5} {sub["cate"].mean():>12.4f} {y_iron:>10.3f} {y_no_iron:>12.3f} {diff_real:>10.3f}')
    
    # Teste: Q5 - Q1 significativo?
    q1_cate = test.loc[test['cate_quintile'] == 'Q1 (menor)', 'cate'].values
    q5_cate = test.loc[test['cate_quintile'] == 'Q5 (maior)', 'cate'].values
    diff_q5_q1 = q5_cate.mean() - q1_cate.mean()
    
    # Bootstrap para diferença Q5-Q1
    boot_diffs = []
    for _ in range(1000):
        b_q5 = np.random.choice(q5_cate, len(q5_cate), replace=True).mean()
        b_q1 = np.random.choice(q1_cate, len(q1_cate), replace=True).mean()
        boot_diffs.append(b_q5 - b_q1)
    ci_lo = np.percentile(boot_diffs, 2.5)
    ci_hi = np.percentile(boot_diffs, 97.5)
    sig = '*' if ci_lo > 0 or ci_hi < 0 else ''
    
    print(f'\n  GATES test (Q5-Q1): {diff_q5_q1:.4f} (95% CI: {ci_lo:.4f} to {ci_hi:.4f}) {sig}')
    print(f'  Heterogeneidade {"CONFIRMADA" if sig else "não significativa"}' )

## 7. SHAP + Correlação com Regressão Epidemiológica

SHAP via surrogate GBR no CATE. Depois: correlação Pearson e Spearman entre SHAP importance e coeficientes da regressão ajustada (como r=0.968 do paper IJMI).

In [ ]:
for outcome_col, outcome_name in OUTCOMES.items():
    print(f"\n{'='*80}")
    print(f'SHAP + VALIDAÇÃO — {outcome_name}')
    print(f"{'='*80}")
    
    r = results_all[outcome_col]
    data = r['data']
    X_all = data[FEATURES].values
    cate_all = r['models']['CausalForestDML']['model'].effect(X_all).flatten()
    
    # Surrogate
    surrogate = GradientBoostingRegressor(n_estimators=300, max_depth=4, random_state=42)
    surrogate.fit(X_all, cate_all)
    r2 = surrogate.score(X_all, cate_all)
    print(f'  Surrogate R²: {r2:.4f}')
    
    explainer = shap.TreeExplainer(surrogate)
    shap_vals = explainer.shap_values(X_all)
    
    # SHAP importance
    shap_importance = np.abs(shap_vals).mean(axis=0)
    shap_ranking = pd.DataFrame({'Feature': FEATURE_NAMES, 'SHAP': shap_importance}).sort_values('SHAP', ascending=False)
    
    print(f'\n  --- Top 10 SHAP Features ---')
    for i, (_, row) in enumerate(shap_ranking.head(10).iterrows()):
        print(f'  {i+1:>2}. {row["Feature"]:<30} SHAP={row["SHAP"]:.4f}')
    
    # Correlação SHAP vs Regressão
    print(f'\n  --- Correlação SHAP vs Regressão Epidemiológica ---')
    # Rodar regressão rápida para pegar coeficientes
    reg_data = data[['iron_any', outcome_col] + FEATURES].dropna()
    X_reg = sm.add_constant(reg_data[['iron_any'] + FEATURES].astype(float))
    y_reg = reg_data[outcome_col].astype(float)
    w_reg = np.ones(len(reg_data))  # sem peso para simplificar
    res_reg = sm.OLS(y_reg, X_reg).fit(cov_type='HC1')
    
    # Coeficientes das features (excluindo constante e iron_any)
    reg_coefs = res_reg.params[FEATURES].abs().values
    
    # Correlação
    r_pearson, p_pearson = pearsonr(shap_importance, reg_coefs)
    r_spearman, p_spearman = spearmanr(shap_importance, reg_coefs)
    
    print(f'  Pearson:  r={r_pearson:.3f} (p={p_pearson:.4f})')
    print(f'  Spearman: r={r_spearman:.3f} (p={p_spearman:.4f})')
    
    # SHAP summary plot
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_vals, features=X_all, feature_names=FEATURE_NAMES,
                      show=False, max_display=15)
    plt.title(f'SHAP — Drivers do efeito do ferro sobre {outcome_name}', fontsize=12)
    plt.tight_layout()
    fname = f'/Users/marcelocarvalhoesilva/project/iron/data_Eval/shap_validated_{outcome_col.replace("vd_","").replace("_final","")}.png'
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.show()
    
    r['shap_importance'] = shap_importance
    r['shap_vals'] = shap_vals

## 8. Equidade com Bootstrap CI

CATE por região, raça, IEN, insegurança alimentar — com IC 95% bootstrap e teste de disparidade.

In [ ]:
for outcome_col, outcome_name in OUTCOMES.items():
    print(f"\n{'='*80}")
    print(f'EQUIDADE — {outcome_name}')
    print(f"{'='*80}")
    
    r = results_all[outcome_col]
    data = r['data'].copy()
    data['cate'] = r['models']['CausalForestDML']['model'].effect(data[FEATURES].values).flatten()
    
    equity_groups = {
        'IEN Q1 (mais pobre)': data['ien'] == 1,
        'IEN Q2': data['ien'] == 2,
        'IEN Q3': data['ien'] == 3,
        'IEN Q4': data['ien'] == 4,
        'IEN Q5 (mais rico)': data['ien'] == 5,
        'Cor preta': data['race_preta'] == 1,
        'Cor parda': data['race_parda'] == 1,
        'Cor branca/outra': (data['race_preta'] == 0) & (data['race_parda'] == 0),
        'Norte': data['reg_norte'] == 1,
        'Nordeste': data['reg_nordeste'] == 1,
        'Sul': data['reg_sul'] == 1,
        'Sudeste': (data['reg_norte']==0) & (data['reg_nordeste']==0) & (data['reg_sul']==0) & (data['reg_co']==0),
        'Centro-Oeste': data['reg_co'] == 1,
        'Inseg. alimentar grave': data['ebia_grave'] == 1,
        'Segurança alimentar': (data['ebia_leve']==0) & (data['ebia_mod']==0) & (data['ebia_grave']==0),
        'Saneamento adequado': data['esgoto_adeq'] == 1,
        'Saneamento inadequado': data['esgoto_adeq'] == 0,
    }
    
    print(f'\n  {"Grupo":<30} {"N":>6} {"CATE":>8} {"95% CI":>22} {"Sig":>4}')
    print(f'  {"-"*70}')
    
    cate_results = []
    for name, mask in equity_groups.items():
        if mask.sum() < 10:
            continue
        cate_sub = data.loc[mask, 'cate'].values
        ci_lo, ci_hi = bootstrap_cate(cate_sub)
        sig = '*' if (ci_lo > 0 or ci_hi < 0) else ''
        cate_results.append({'name': name, 'n': mask.sum(), 'cate': cate_sub.mean(), 'ci_lo': ci_lo, 'ci_hi': ci_hi})
        print(f'  {name:<30} {mask.sum():>6} {cate_sub.mean():>8.4f} ({ci_lo:>8.4f} to {ci_hi:>8.4f}) {sig:>4}')
    
    # Disparidade máxima por IEN
    ien_cates = [c for c in cate_results if c['name'].startswith('IEN')]
    if len(ien_cates) >= 2:
        max_c = max(ien_cates, key=lambda x: x['cate'])
        min_c = min(ien_cates, key=lambda x: x['cate'])
        disp = max_c['cate'] - min_c['cate']
        print(f'\n  Disparidade IEN (max-min): {disp:.4f} ({max_c["name"]} vs {min_c["name"]})')

## 9. Trade-off Scatter + Modelo de Decisão Clínica

In [ ]:
# Juntar CATEs de WFL e Hb
data_wfl = results_all['vd_anthro_zwfl']['data'].copy()
data_hb = results_all['vd_hb_final']['data'].copy()

data_wfl['cate_wfl'] = results_all['vd_anthro_zwfl']['models']['CausalForestDML']['model'].effect(data_wfl[FEATURES].values).flatten()
data_hb['cate_hb'] = results_all['vd_hb_final']['models']['CausalForestDML']['model'].effect(data_hb[FEATURES].values).flatten()

common = data_wfl.index.intersection(data_hb.index)
scatter = pd.DataFrame({
    'cate_wfl': data_wfl.loc[common, 'cate_wfl'].values,
    'cate_hb': data_hb.loc[common, 'cate_hb'].values,
    'breastfed': data_wfl.loc[common, 'breastfed'].values,
    'age_months': data_wfl.loc[common, 'age_months'].values,
    'ien': data_wfl.loc[common, 'ien'].values,
})

# Classificar decisão
def classify_decision(row):
    if row['cate_hb'] > 0 and row['cate_wfl'] >= -0.05:
        return 'SUPLEMENTAR'
    elif row['cate_hb'] > 0 and row['cate_wfl'] < -0.05:
        return 'TRADE-OFF (avaliar)'
    elif row['cate_hb'] <= 0:
        return 'NÃO SUPLEMENTAR'
    return 'AVALIAR'

scatter['decision'] = scatter.apply(classify_decision, axis=1)

print('=== DECISÃO CLÍNICA BASEADA NO CAUSAL FOREST ===')
print(scatter['decision'].value_counts())
print(f'\n--- Por aleitamento ---')
print(pd.crosstab(scatter['breastfed'].map({0: 'Não amam', 1: 'Amamentado'}), scatter['decision']))

# Scatter
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors = {'SUPLEMENTAR': 'green', 'TRADE-OFF (avaliar)': 'orange', 'NÃO SUPLEMENTAR': 'red', 'AVALIAR': 'gray'}
for dec, color in colors.items():
    sub = scatter[scatter['decision'] == dec]
    if len(sub) > 0:
        axes[0].scatter(sub['cate_hb'], sub['cate_wfl'], alpha=0.4, c=color, label=dec, s=20)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('CATE Hemoglobina (g/dL)')
axes[0].set_ylabel('CATE Z peso/comprimento')
axes[0].set_title('Decisão Clínica Individual')
axes[0].legend(fontsize=9)

for bf, color, label in [(0, 'red', 'Não amamentado'), (1, 'blue', 'Amamentado')]:
    sub = scatter[scatter['breastfed'] == bf]
    axes[1].scatter(sub['cate_hb'], sub['cate_wfl'], alpha=0.3, c=color, label=label, s=20)
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('CATE Hemoglobina (g/dL)')
axes[1].set_ylabel('CATE Z peso/comprimento')
axes[1].set_title('Por Status de Aleitamento')
axes[1].legend(fontsize=11)

plt.suptitle('Causal Forest: Decisão individualizada sobre suplementação de ferro', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/Users/marcelocarvalhoesilva/project/iron/data_Eval/clinical_decision.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Modelo para Deploy (GitHub Pages)

Exporta o modelo como um preditor simples: recebe características da criança, retorna recomendação de suplementação.

In [ ]:
import json
import joblib

# Treinar surrogate final nos dados completos (para deploy)
# Usa o surrogate GBR que prediz CATE a partir das features

# WFL surrogate
data_all_wfl = results_all['vd_anthro_zwfl']['data']
X_deploy = data_all_wfl[FEATURES].values
cate_wfl_all = results_all['vd_anthro_zwfl']['models']['CausalForestDML']['model'].effect(X_deploy).flatten()

# Hb surrogate
data_all_hb = results_all['vd_hb_final']['data']
X_deploy_hb = data_all_hb[FEATURES].values
cate_hb_all = results_all['vd_hb_final']['models']['CausalForestDML']['model'].effect(X_deploy_hb).flatten()

# Treinar surrogates
surrogate_wfl = GradientBoostingRegressor(n_estimators=300, max_depth=4, random_state=42)
surrogate_wfl.fit(X_deploy, cate_wfl_all)

surrogate_hb = GradientBoostingRegressor(n_estimators=300, max_depth=4, random_state=42)
surrogate_hb.fit(X_deploy_hb, cate_hb_all)

# Salvar modelos
deploy_dir = '/Users/marcelocarvalhoesilva/project/iron/deploy'
import os
os.makedirs(deploy_dir, exist_ok=True)

joblib.dump(surrogate_wfl, f'{deploy_dir}/surrogate_wfl.joblib')
joblib.dump(surrogate_hb, f'{deploy_dir}/surrogate_hb.joblib')

# Salvar feature names e decisão logic
deploy_config = {
    'features': FEATURES,
    'feature_names': FEATURE_NAMES,
    'feature_descriptions': {
        'age_months': {'type': 'number', 'label': 'Idade em meses', 'min': 6, 'max': 18},
        'male': {'type': 'binary', 'label': 'Sexo masculino'},
        'race_preta': {'type': 'binary', 'label': 'Cor preta'},
        'race_parda': {'type': 'binary', 'label': 'Cor parda'},
        'reg_norte': {'type': 'binary', 'label': 'Região Norte'},
        'reg_nordeste': {'type': 'binary', 'label': 'Região Nordeste'},
        'reg_sul': {'type': 'binary', 'label': 'Região Sul'},
        'reg_co': {'type': 'binary', 'label': 'Região Centro-Oeste'},
        'urban': {'type': 'binary', 'label': 'Zona urbana'},
        'educ_fund_comp': {'type': 'binary', 'label': 'Mãe: fundamental completo'},
        'educ_medio': {'type': 'binary', 'label': 'Mãe: ensino médio'},
        'educ_superior': {'type': 'binary', 'label': 'Mãe: ensino superior'},
        'ien': {'type': 'number', 'label': 'IEN quintil (1=pobre, 5=rico)', 'min': 1, 'max': 5},
        'ebia_leve': {'type': 'binary', 'label': 'Inseg. alimentar leve'},
        'ebia_mod': {'type': 'binary', 'label': 'Inseg. alimentar moderada'},
        'ebia_grave': {'type': 'binary', 'label': 'Inseg. alimentar grave'},
        'esgoto_adeq': {'type': 'binary', 'label': 'Saneamento adequado'},
        'agua_rede': {'type': 'binary', 'label': 'Água de rede'},
        'cesarean': {'type': 'binary', 'label': 'Parto cesariana'},
        'gest_weeks': {'type': 'number', 'label': 'Idade gestacional (semanas)', 'min': 26, 'max': 43},
        'birth_weight': {'type': 'number', 'label': 'Peso ao nascer (g)', 'min': 500, 'max': 5500},
        'prenatal_visits': {'type': 'number', 'label': 'Consultas pré-natal', 'min': 0, 'max': 20},
        'maternal_age': {'type': 'number', 'label': 'Idade materna (anos)', 'min': 13, 'max': 50},
        'breastfed': {'type': 'binary', 'label': 'Em aleitamento materno'},
        'formula': {'type': 'binary', 'label': 'Usa fórmula infantil'},
        'feed_exclusive': {'type': 'binary', 'label': 'Aleitamento exclusivo'},
        'feed_mixed': {'type': 'binary', 'label': 'Aleitamento misto'},
    },
    'decision_rules': {
        'SUPLEMENTAR': 'CATE Hb > 0 E CATE WFL >= -0.05',
        'TRADE-OFF': 'CATE Hb > 0 E CATE WFL < -0.05',
        'NAO_SUPLEMENTAR': 'CATE Hb <= 0',
    }
}

with open(f'{deploy_dir}/config.json', 'w') as f:
    json.dump(deploy_config, f, indent=2, ensure_ascii=False)

print(f'Modelos salvos em {deploy_dir}/')
print(f'  surrogate_wfl.joblib: {os.path.getsize(f"{deploy_dir}/surrogate_wfl.joblib")/1024:.0f} KB')
print(f'  surrogate_hb.joblib: {os.path.getsize(f"{deploy_dir}/surrogate_hb.joblib")/1024:.0f} KB')
print(f'  config.json: {os.path.getsize(f"{deploy_dir}/config.json")/1024:.0f} KB')

# Demo: predição para uma criança exemplo
print(f'\n--- DEMO: Predição para uma criança ---')
exemplo = {
    'age_months': 8, 'male': 1, 'race_preta': 0, 'race_parda': 1,
    'reg_norte': 0, 'reg_nordeste': 1, 'reg_sul': 0, 'reg_co': 0, 'urban': 1,
    'educ_fund_comp': 0, 'educ_medio': 1, 'educ_superior': 0, 'ien': 2,
    'ebia_leve': 1, 'ebia_mod': 0, 'ebia_grave': 0,
    'esgoto_adeq': 1, 'agua_rede': 1, 'cesarean': 0,
    'gest_weeks': 39, 'birth_weight': 3200, 'prenatal_visits': 7, 'maternal_age': 25,
    'breastfed': 1, 'formula': 0, 'feed_exclusive': 1, 'feed_mixed': 0,
}
x = np.array([[exemplo[f] for f in FEATURES]])
pred_hb = surrogate_hb.predict(x)[0]
pred_wfl = surrogate_wfl.predict(x)[0]

if pred_hb > 0 and pred_wfl >= -0.05:
    decision = 'SUPLEMENTAR'
elif pred_hb > 0 and pred_wfl < -0.05:
    decision = 'TRADE-OFF (avaliar caso a caso)'
else:
    decision = 'NÃO SUPLEMENTAR'

print(f'  Menino, 8m, NE, pardo, amam. exclusivo, IEN Q2')
print(f'  CATE Hb: {pred_hb:+.4f} g/dL')
print(f'  CATE WFL: {pred_wfl:+.4f} z-score')
print(f'  → Decisão: {decision}')